In [1]:
import sys 

#==== method specific ==== 
import networkx as nx
import scglue
from itertools import chain
import seaborn as sns
from matplotlib import rcParams
import h5py

#from matplotlib import rcParams
from anndata import AnnData
import anndata as ad
import scipy
import numpy as np
import pandas as pd
import scipy.io as sio
import os
import scanpy as sc
from copy import deepcopy
#from utils_eval import read_mtx_folder, write_adata
import timeit

from sklearn.metrics import adjusted_rand_score as ari_score
from sklearn.metrics.cluster import normalized_mutual_info_score as nmi_score
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, adjusted_mutual_info_score
from sklearn.metrics import fowlkes_mallows_score, homogeneity_score, completeness_score, v_measure_score, silhouette_score
from sklearn.cluster import KMeans
import anndata as ad

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

/data1/chengyue/miniconda3/envs/glue/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data1/chengyue/miniconda3/envs/glue/lib/python3.8/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [2]:
def read_mtx_folder(dir_path,mod_key,var_list,obs_list):
    mtx_path = []
    tsv_path = []
    for file in os.listdir(dir_path):
        if file.endswith(".mtx"):
            mtx_path.append(os.path.join(dir_path, file))
    if(len(mtx_path)==0):
        print("no .mtx file found, exiting function")
        return
    counts = sio.mmread(mtx_path[0])
    #if type(counts) != "scipy.sparse.csr.csr_matrix": counts = counts.tocsr()
    if not isinstance(counts, scipy.sparse.csr_matrix): 
        counts = counts.tocsr()
    
    var_v = []
    for i in var_list:
        var_v.append(pd.read_csv(os.path.join(dir_path,i+".csv"), sep = '\t'))
    var_df = pd.concat(var_v, axis=1)
    #var_df = var_df.set_axis(var_list, axis='columns')
    
    obs_v = []
    for i in obs_list:
        obs_v.append(pd.read_csv(os.path.join(dir_path,i+".csv"), sep = '\t'))
    obs_df = pd.concat(obs_v, axis=1)
    obs_df = obs_df.set_axis(obs_list, axis='columns')
    if(counts.shape[0]!=obs_df.shape[0]): counts=deepcopy(counts.transpose())
    adata = AnnData(counts,obs=obs_df,var=var_df)
    adata.obs = adata.obs.set_axis(list(adata.obs["barcodes"]),axis="index")
    adata.var= adata.var.set_axis(list(adata.var['feature_name']),axis="index")
    adata.var["modality"] = mod_key
    return(adata)

def eva(y_true, y_pred):
    nmi = nmi_score(y_true, y_pred)
    ari = ari_score(y_true, y_pred)
    ami = adjusted_mutual_info_score(y_true, y_pred)
    fmi = fowlkes_mallows_score(y_true, y_pred)
    hom = homogeneity_score(y_true, y_pred)
    com = completeness_score(y_true, y_pred)
    v = v_measure_score(y_true, y_pred)
    return nmi, ari, ami, fmi, hom, com, v

def eva_nolabel(data, y):
    db = davies_bouldin_score(data, y)
    ch = calinski_harabasz_score(data, y)
    asw = silhouette_score(data, y)
    return db, ch, asw

In [ ]:
def run_glue_fn(in_dir, out_dir):#in_dir,out_dir,label_dir=None):
    start = timeit.default_timer()

    rna = read_mtx_folder(os.path.join(in_dir,"paired_RNA/"),
                                       "Gene Expression",
                                       ["gene"],
                                       ["barcodes"])
    
    atac = read_mtx_folder(os.path.join(in_dir,"paired_ATAC/"),
                                       "Peaks",
                                       ["peak"],
                                       ["barcodes"])
    
    cell_name = pd.read_csv('./human_brain_3k/label.csv', usecols=[1])
    cell_type, y = np.unique(cell_name, return_inverse=True)
    cluster_number = int(max(y) - min(y) + 1)   
    
    rna.obs['dataset'] = 'multiomeRNA'
    atac.obs['dataset'] = 'multiomeATAC'

    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(os.path.join(out_dir,"glue"), exist_ok=True)
    os.makedirs(os.path.join(out_dir,"runtime"), exist_ok=True)
    
    # preprocessing of scRNA
    rna.layers["counts"] = rna.X.copy()
    sc.pp.filter_genes(rna, min_cells=3)
    sc.pp.highly_variable_genes(rna, n_top_genes=2000, flavor="seurat_v3")
    sc.pp.normalize_total(rna)
    sc.pp.log1p(rna)
    sc.pp.scale(rna)
    sc.tl.pca(rna, n_comps=100, svd_solver="auto")
    
    # preprocessing of snATAC
    sc.pp.filter_genes(atac,min_counts=1)
    scglue.data.lsi(atac, n_components=100, n_iter=15)

    # build graph
    scglue.data.get_gene_annotation(
        # this works for human hg38 genome-build 
        rna, gtf="./gencode.v47.annotation.gtf", #  "/home/chengyue/GLUE-master/refdata-gex-mm10-2020-A/genes/genes.gtf" "/home/chengyue/GLUE-master/gencode.v47.annotation.gtf.gz"
        gtf_by="gene_name"
    )
    rna.var.loc[:, ["chrom", "chromStart", "chromEnd"]].head()
    
    split = atac.var_names.str.split(r"[:-]")
    atac.var["chrom"] = split.map(lambda x: x[0])
    atac.var["chromStart"] = split.map(lambda x: x[1]).astype(int)
    atac.var["chromEnd"] = split.map(lambda x: x[2]).astype(int)
    
    print(atac.var)
    atac_chrs = atac.var['chrom'].value_counts().index.tolist()
    row_keep = rna.var_names[rna.var['chrom'].isin(atac_chrs).tolist()]
    row_keep = list(set(row_keep))
    #rna = rna.loc[:, row_keep].copy()
    rna.var_names_make_unique()
    rna = rna[:,row_keep].copy()
    guidance = scglue.genomics.rna_anchored_guidance_graph(rna, atac)
    scglue.graph.check_graph(guidance, [rna, atac])
    
    # prepare for training 
    scglue.models.configure_dataset(
        rna, "NB", use_highly_variable=True,
        use_layer="counts", use_rep="X_pca"
    )
    scglue.models.configure_dataset(
        atac, "NB", use_highly_variable=True,
        use_rep="X_lsi"#X_lsi
    )

    guidance_hvf = guidance.subgraph(chain(
        rna.var.query("highly_variable").index,
        atac.var.query("highly_variable").index
    )).copy()
    
    # GLUE training 
    glue = scglue.models.fit_SCGLUE(
        {"rna": rna, "atac": atac}, guidance_hvf,
        fit_kws={"directory": os.path.join(out_dir,"glue")}
    )
    
    dx = scglue.models.integration_consistency(
        glue, {"rna": rna, "atac": atac}, guidance_hvf
    )
    print(dx)
    rna.obsm["X_glue"] = glue.encode_data("rna", rna)
    atac.obsm["X_glue"] = glue.encode_data("atac", atac)
    combined = ad.concat([rna, atac])

    # extract latent representation
    res_df = pd.DataFrame(combined.obsm['X_glue'],index=combined.obs.index)
    # set column names as latent_x 
    res_df = res_df.set_axis(["latent_" + s  for s in res_df.columns.astype("str").tolist()],axis="columns")
    res_df['dataset'] = combined.obs['dataset']
    res_df['dataset'] = res_df['dataset'].astype("string")
    
    # save latent representation and model
    
    csv_out = os.path.join(out_dir, "glue","glue_result.csv")
    res_df.to_csv(csv_out)
    model_out = os.path.join(out_dir,"glue","glue.dill")
    glue.save(model_out)
    stop = timeit.default_timer()
    
    n_clusters = cluster_number

    kmeans = KMeans(n_clusters = n_clusters, n_init=20)
    y_pred_z = kmeans.fit_predict(rna.obsm["X_glue"])   
    rna_z = os.path.join(out_dir,"glue","rna_z.csv")
    rna_pred = os.path.join(out_dir,"glue","rna_pred.csv")
    np.savetxt(rna_pred, y_pred_z)
    np.savetxt(rna_z, rna.obsm["X_glue"])
    print(y_pred_z)
    nmi, ari, ami, fmi, hom, com, v = eva(y, y_pred_z)
    
    db, ch, asw = eva_nolabel(rna.obsm["X_glue"], y_pred_z)
    print('RNA: nmi, ari, ami, fmi, hom, com, v:', nmi, ari, ami, fmi, hom, com, v)
    print('RNA: db, ch, asw:', db, ch, asw)

    kmeans = KMeans(n_clusters = n_clusters, n_init=20)
    y_pred_z = kmeans.fit_predict(atac.obsm["X_glue"])   
    atac_z = os.path.join(out_dir,"glue","atac_z.csv")
    atac_pred = os.path.join(out_dir,"glue","atac_pred.csv")
    np.savetxt(atac_pred, y_pred_z)
    np.savetxt(atac_z, atac.obsm["X_glue"])
    nmi, ari, ami, fmi, hom, com, v = eva(y, y_pred_z)
    
    db, ch, asw = eva_nolabel(atac.obsm["X_glue"], y_pred_z)
    print('ATAC: nmi, ari, ami, fmi, hom, com, v:', nmi, ari, ami, fmi, hom, com, v)
    print('ATAC: db, ch, asw:', db, ch, asw)
    print('Time(s): ', stop - start)  
    # record time 
    runtime_out = os.path.join(out_dir,"runtime","glue_runtime.txt")
    print(stop - start,  file=open(runtime_out, 'w'))
    print("------ Done ------")

In [ ]:
run_glue_fn('./human_brain_3k/', './human_brain_3k/glue-output/')

                                      features  n_counts       chrom  \
chr1:180995-181723          chr1:180995-181723     123.0        chr1   
chr1:191025-191934          chr1:191025-191934    1884.0        chr1   
chr1:629469-630395          chr1:629469-630395     889.0        chr1   
chr1:633551-634475          chr1:633551-634475    3714.0        chr1   
chr1:778288-779198          chr1:778288-779198    2918.0        chr1   
...                                        ...       ...         ...   
KI270713.1:21467-22401  KI270713.1:21467-22401    1892.0  KI270713.1   
KI270713.1:25956-26766  KI270713.1:25956-26766      75.0  KI270713.1   
KI270713.1:29714-30467  KI270713.1:29714-30467     150.0  KI270713.1   
KI270713.1:31270-32183  KI270713.1:31270-32183     421.0  KI270713.1   
KI270713.1:33235-33800  KI270713.1:33235-33800      62.0  KI270713.1   

                        chromStart  chromEnd  
chr1:180995-181723          180995    181723  
chr1:191025-191934          191025    191

window_graph: 100%|██████████| 19232/19232 [00:01<00:00, 10905.53it/s]


[INFO] check_graph: Checking variable coverage...
[INFO] check_graph: Checking edge attributes...
[INFO] check_graph: Checking self-loops...
[INFO] check_graph: Checking graph symmetry...
[INFO] check_graph: All checks passed!
[INFO] fit_SCGLUE: Pretraining SCGLUE model...
[INFO] autodevice: Using GPU 3 as computation device.
[INFO] check_graph: Checking variable coverage...
[INFO] check_graph: Checking edge attributes...
[INFO] check_graph: Checking self-loops...
[INFO] check_graph: Checking graph symmetry...
[INFO] SCGLUEModel: Setting `graph_batch_size` = 17309
[INFO] SCGLUEModel: Setting `max_epochs` = 528
[INFO] SCGLUEModel: Setting `patience` = 44
[INFO] SCGLUEModel: Setting `reduce_lr_patience` = 22
[INFO] SCGLUETrainer: Using training directory: "/data1/chengyue/gm/fig2/human_brain_3k/glue-output/glue/pretrain"


/data1/chengyue/miniconda3/envs/glue/lib/python3.8/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


[INFO] SCGLUETrainer: [Epoch 10] train={'g_nll': 0.529, 'g_kl': 0.005, 'g_elbo': 0.534, 'x_rna_nll': 0.465, 'x_rna_kl': 0.02, 'x_rna_elbo': 0.485, 'x_atac_nll': 0.266, 'x_atac_kl': 0.003, 'x_atac_elbo': 0.269, 'dsc_loss': 0.675, 'vae_loss': 0.776, 'gen_loss': 0.742}, val={'g_nll': 0.53, 'g_kl': 0.005, 'g_elbo': 0.534, 'x_rna_nll': 0.528, 'x_rna_kl': 0.019, 'x_rna_elbo': 0.546, 'x_atac_nll': 0.296, 'x_atac_kl': 0.003, 'x_atac_elbo': 0.299, 'dsc_loss': 0.705, 'vae_loss': 0.867, 'gen_loss': 0.832}, 1.0s elapsed
[INFO] SCGLUETrainer: [Epoch 20] train={'g_nll': 0.517, 'g_kl': 0.008, 'g_elbo': 0.525, 'x_rna_nll': 0.445, 'x_rna_kl': 0.014, 'x_rna_elbo': 0.459, 'x_atac_nll': 0.252, 'x_atac_kl': 0.002, 'x_atac_elbo': 0.254, 'dsc_loss': 0.688, 'vae_loss': 0.734, 'gen_loss': 0.7}, val={'g_nll': 0.516, 'g_kl': 0.008, 'g_elbo': 0.524, 'x_rna_nll': 0.453, 'x_rna_kl': 0.013, 'x_rna_elbo': 0.466, 'x_atac_nll': 0.268, 'x_atac_kl': 0.002, 'x_atac_elbo': 0.27, 'dsc_loss': 0.696, 'vae_loss': 0.757, 'gen_l

...

[INFO] get_metacells: Aggregating metacells...
[INFO] metacell_corr: Computing correlation on 100 common metacells...
[INFO] get_metacells: Clustering metacells...


/data1/chengyue/miniconda3/envs/glue/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
[WARNING] get_metacells: `faiss` is not installed, using `sklearn` instead... This might be slow with a large number of cells. Consider installing `faiss` following the guide from https://github.com/facebookresearch/faiss/blob/main/INSTALL.md
/data1/chengyue/miniconda3/envs/glue/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


[INFO] get_metacells: Aggregating metacells...
[INFO] metacell_corr: Computing correlation on 194 common metacells...
   n_meta  consistency
0      10     0.511691
1      20     0.426044
2      50     0.371726
3     100     0.307829
4     200     0.260488


/data1/chengyue/miniconda3/envs/glue/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/data1/chengyue/miniconda3/envs/glue/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/data1/chengyue/miniconda3/envs/glue/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


[0 3 3 ... 4 0 5]
RNA: nmi, ari, ami, fmi, hom, com, v: 0.5780474543927545 0.3217296998093281 0.5754177571431 0.4382386952203957 0.538077699510934 0.6244318080479487 0.5780474543927545
RNA: db, ch, asw: 1.623286754247913 460.7945621160537 0.239299
ATAC: nmi, ari, ami, fmi, hom, com, v: 0.5840437776327023 0.27850390430307653 0.5812596644802931 0.4554504350084948 0.5078990265280215 0.6870465053728605 0.5840437776327023
ATAC: db, ch, asw: 1.5776080267396853 478.2732421582265 0.4307584
Time(s):  1748.2553352415562
------ Done ------
